In [1]:
45

142+1


143

In [2]:
import os
from dotenv import load_dotenv

# LANGCHAIN
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

C:\Users\ashwi\AppData\Local\Temp\ipykernel_10320\1113816932.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
load_dotenv()


True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
embeeding_key= os.getenv("EMBEEDING_API_KEY")


In [5]:
print("env_variable_loaded")

env_variable_loaded


LOADING OUR DATA


In [6]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

DATA INGESTION

In [7]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")

documents = loader.load()
print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### LANGCHAIN DOCUMENT 
langchain processes is everything in form of documents


DOCUMENTS :

PAGE CONTENT -- THE ACTUAL DATA
METADATA -- EXTRA INFORMATION ABOUT THE DATA

In [8]:
len(documents)

1

In [9]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [10]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


SPILTING OUR DATA


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spiltters = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = text_spiltters.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [12]:
len(chunks)

9

NOW EACH SPILTTED CHUNK IS DOC 


In [13]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [14]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [15]:
print(chunks[7])

page_content='7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.' metadata={'source': 'data\\hr_policy.txt'}


In [16]:
print(chunks[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)


NOW EMBEED THE DATA 


In [17]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)

In [19]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)
print("EMB MODEL READY THE NAME IS :" , embeddings_model.model_name)

EMB MODEL READY THE NAME IS : jina-embeddings-v2-base-en


STORE THE DATA IN VECTOR DB

In [20]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED ", vector_store.index.ntotal )

CHUNKS ARE STORED  9


WE ARE NEVER STORED it(vector db)

In [21]:
test_query = "How many sick leaves employees"

# SIMILARITY SEARCH
top_matches = vector_store.similarity_search(test_query, k=2)

print(f"Query: {test_query}\n")

for i, match in enumerate(top_matches, start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



TOOL

In [30]:
retriever = vector_store.as_retriever(search_kwargs={"k":3}) #return top 3 relevant information
def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.

    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL 


LLM

In [22]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature =0  # creativity of model
  )

llm.model_name

'openai/gpt-oss-120b'

In [23]:
test_tresponse = llm.invoke("Hey is learning rag hard? in a joke way in 1 line")

test_tresponse.content

'Learning RAG is like trying to teach a cat to fetch—possible, but you’ll spend most of the time chasing after the data!'

AI AGENT

3--

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - no memory

In [31]:
from langchain.agents import create_agent

In [32]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt= """
      You are a friendly HR assistant.
      Always use the search_hr_policy tool to look up
      facts before answering 
      If the answer isn't in the search results , say you don't know "
      Instead of guessing."
"""
)

print("HR assistant agent is ready to answer")

HR assistant agent is ready to answer


In [33]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [34]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [35]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='5ed674fb-4b58-488b-a224-5a2a7647dc29'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". As an HR assistant, we need to answer. The developer instructions: "Always use the search_hr_policy tool to look up facts before answering. If the answer isn\'t in the search results, say you don\'t know instead of guessing." The question is about which organization the assistant works for. This is not about HR policy. The policy likely doesn\'t contain that. So we need to use the tool to search? The instruction says always use the search_hr_policy tool to look up facts before answering. So we should search for something like "organization" or "company name". Let\'s try search.', 'tool_calls': [{'id': 'fc_4c0185e7-fa7c-48bb-ac9a-0f6b022b43be', 'function': {'arguments': '{"question":"Which organization does the HR assist

SYSTEM MESSAGE - HR ASSISANT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THSES ARE THE POLICIES 

In [36]:
response["messages"][-2].content

'COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'

In [37]:
print(response["messages"][-1].content)

I’m part of the HR team at **Acme Corp**.
